# 1. Описание задачи

## Бизнес-контекст

Разрабатывается мобильный мессенджер с функцией умных подсказок. Во время набора сообщения приложение должно предлагать пользователю наиболее вероятное следующее слово.

Примеры работы функции:

| Введённый текст | Возможное следующее слово |
|---|---|
| `the president of the` | `united` |
| `i want to` | `go` |
| `the weather today is` | `sunny` |
| `new york is a` | `city` |

Такая функция применяется в мобильных клавиатурах, мессенджерах, поисковых строках и голосовых помощниках.

## Цель проекта

Цель проекта — обучить и сравнить несколько архитектур нейронных сетей для предсказания следующего слова, а затем определить, какая из них лучше подходит для использования на мобильном устройстве.

При выборе модели необходимо учитывать:

- качество предсказаний;
- количество параметров;
- размер модели;
- скорость обучения;
- скорость получения предсказания.

## Постановка задачи

Модель получает последовательность токенов и должна предсказать следующий токен.

Например:

```text
Вход:    the cat sat on the
Таргет:  cat sat on the mat

Таким образом, задача предсказания следующего слова рассматривается как задача многоклассовой классификации. Модель рассчитывает вероятности всех слов из словаря, а правильным классом является фактическое следующее слово.

In [1]:
%pip install -v numpy pandas matplotlib seaborn tqdm datasets transformers

Using pip 26.2.1 from d:\МИФИ_учеба\3_семестр\02_Работа_с_изображениями_и_текстами\01_проект\.venv\Lib\site-packages\pip (python 3.14)
Note: you may need to restart the kernel to use updated packages.


In [2]:
# %pip install --upgrade --force-reinstall torch --index-url https://download.pytorch.org/whl/cu132

In [3]:
# %pip install "fsspec[http]==2026.6.0"

In [4]:
import sys
import math
import time
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from tqdm.auto import tqdm

from transformers import AutoTokenizer, AutoModelForCausalLM

d:\МИФИ_учеба\3_семестр\02_Работа_с_изображениями_и_текстами\01_проект\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
SEED = 42


def set_seed(seed: int = 42) -> None:
    """Фиксирует генераторы случайных чисел."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

In [6]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# True — быстрый локальный запуск
# False — итоговый запуск на полном датасете
FAST_RUN = True

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"Устройство: {DEVICE}")
print(f"Быстрый режим: {FAST_RUN}")

if torch.cuda.is_available():
    print(f"Видеокарта: {torch.cuda.get_device_name(0)}")

Python: 3.14.3
PyTorch: 2.14.1+cu132
Устройство: cuda
Быстрый режим: True
Видеокарта: NVIDIA GeForce RTX 5060 Ti


# 3. Загрузка WikiText-2

Для обучения и оценки моделей используется датасет **WikiText-2** — набор англоязычных текстов из статей Википедии.

Датасет имеет стандартное разделение:

- `train` — обучение моделей;
- `validation` — оценка качества после каждой эпохи;
- `test` — итоговая проверка лучших моделей.

На этом этапе загрузим данные, проверим их структуру и выведем несколько примеров. Обучающая, валидационная и тестовая части не будут смешиваться.

In [10]:
DATASET_NAME ="Salesforce/wikitext"
DATASET_CONFIG = "wikitext-2-raw-v1"

dataset = load_dataset(
    DATASET_NAME,
    DATASET_CONFIG
)

dataset

Generating validation split: 100%|██████████| 3760/3760 [00:00<00:00, 891345.90 examples/s]


DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})

# 4. Анализ данных

Проведём разведочный анализ WikiText-2. Определим количество пустых и непустых текстовых блоков, приблизительное число слов и распределение длин текстов.

Это позволит оценить объём данных и выбрать подходящие параметры для формирования обучающих последовательностей.

In [11]:
required_splits = {"train", "validation", "test"}

assert required_splits.issubset(dataset.keys()), (
    "В датасете отсутствует один из обязательных разделов"
)

for split_name in ["train", "validation", "test"]:
    split = dataset[split_name]

    print(f"{split_name}:")
    print(f"  количество строк: {len(split):,}")
    print(f"  столбцы: {split.column_names}")

train:
  количество строк: 36,718
  столбцы: ['text']
validation:
  количество строк: 3,760
  столбцы: ['text']
test:
  количество строк: 4,358
  столбцы: ['text']


In [12]:
split_sizes = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "rows": [
            len(dataset["train"]),
            len(dataset["validation"]),
            len(dataset["test"]),
        ],
    }
)

split_sizes

,split,rows
0,train,36718
1,validation,3760
2,test,4358


Датасет успешно загружен и содержит три обязательные части:

- обучающая выборка - 36 718 строк;
- валидационная выборка - 3 760 строк;
- тестовая выборка - 4 358 строк.

Каждая часть содержит один столбец `text` с текстовыми блоками. Обучающая выборка является самой большой, поскольку используется для настройки параметров моделей. Валидационная выборка будет применяться для контроля качества во время обучения, а тестовая — для окончательной оценки лучших моделей.

Количество строк не соответствует количеству слов или предложений: одна строка может быть заголовком, абзацем либо быть пустой. Поэтому фактический объём текстовых данных будет определён на этапе разведочного анализа.

In [13]:
def show_text_examples(split, number_of_examples=5):
    """Выводит несколько непустых текстов из выбранной части датасета."""
    
    shown = 0

    for row in split:
        text = row["text"].strip()

        if not text:
            continue

        shown += 1
        print(f"Пример {shown}:")
        print(text[:500])
        print("-" * 80)

        if shown == number_of_examples:
            break


show_text_examples(
    dataset["train"],
    number_of_examples=5
)

Пример 1:
= Valkyria Chronicles III =
--------------------------------------------------------------------------------
Пример 2:
Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and fo
--------------------------------------------------------------------------------
Пример 3:
The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the game more forgiving for series newcomers . Characte

### Первичная проверка данных

Датасет успешно загружен. В обучающей выборке присутствуют:

- заголовки статей;
- заголовки разделов;
- обычные текстовые абзацы;
- пустые строки;
- специальные последовательности для некоторых знаков препинания, например `@-@`.


In [15]:
dataset_statistics = []

for split_name in ["train", "validation", "test"]:
    texts = dataset[split_name]["text"]

    non_empty_texts = [
        text for text in texts
        if text.strip()
    ]

    number_of_words = sum(
        len(text.split())
        for text in non_empty_texts
    )

    number_of_characters = sum(
        len(text)
        for text in non_empty_texts
    )

    dataset_statistics.append(
        {
            "split": split_name,
            "all_rows": len(texts),
            "non_empty_rows": len(non_empty_texts),
            "empty_rows": len(texts) - len(non_empty_texts),
            "approximate_words": number_of_words,
            "characters": number_of_characters,
        }
    )

statistics_df = pd.DataFrame(dataset_statistics)
statistics_df

,split,all_rows,non_empty_rows,empty_rows,approximate_words,characters
0,train,36718,23767,12951,2051910,10892990
1,validation,3760,2461,1299,213886,1142150
2,test,4358,2891,1467,241211,1285622


{'text': ''}
